In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

print("E-Commerce Data Engineering Project Started!")

E-Commerce Data Engineering Project Started!


In [0]:
customers_data = [
    (1, "Arif", "Kolkata"),
    (2, "Priya", "Delhi"),
    (3, "Rahul", "Mumbai"),
    (4, "Sneha", "Kolkata"),
    (5, "Amit", "Bangalore"),
    (2, "Priya", "Delhi"),
    (6, None, "Hyderabad")
]

customers_columns = ["customer_id", "customer_name", "city"]

customers_df = spark.createDataFrame(
    customers_data, customers_columns
)

display(customers_df)

customer_id,customer_name,city
1,Arif,Kolkata
2,Priya,Delhi
3,Rahul,Mumbai
4,Sneha,Kolkata
5,Amit,Bangalore
2,Priya,Delhi
6,null,Hyderabad


In [0]:
products_data = [
    (101, "Laptop", "Electronics", 55000),
    (102, "Mobile", "Electronics", 20000),
    (103, "Headphones", "Accessories", 1500),
    (104, "Keyboard", "Accessories", 800),
    (105, "Monitor", "Electronics", 12000),
    (103, "Headphones", "Accessories", 1500)
]

products_columns = [
    "product_id",
    "product_name",
    "category",
    "price"
]

products_df = spark.createDataFrame(
    products_data, products_columns
)

display(products_df)

product_id,product_name,category,price
101,Laptop,Electronics,55000
102,Mobile,Electronics,20000
103,Headphones,Accessories,1500
104,Keyboard,Accessories,800
105,Monitor,Electronics,12000
103,Headphones,Accessories,1500


In [0]:
orders_data = [
    (1001, 1, 101, 1, "2026-10-01", "Completed"),
    (1002, 2, 102, 2, "2026-10-02", "Completed"),
    (1003, 3, 103, 3, "2026-10-03", "Pending"),
    (1004, 1, 104, 2, "2026-10-04", "Completed"),
    (1005, 4, 105, 1, "2026-10-05", "Cancelled"),
    (1006, 5, 101, 1, "2026-10-06", "Completed"),
    (1007, 2, 103, 2, "2026-10-07", "Completed"),
    (1008, 6, 102, 1, "2026-10-08", "Pending"),
    (1009, 3, 104, 3, "2026-10-09", "Completed"),
    (1002, 2, 102, 2, "2026-10-02", "Completed")
]

orders_columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_date",
    "status"
]

orders_df = spark.createDataFrame(
    orders_data, orders_columns
)

display(orders_df)

order_id,customer_id,product_id,quantity,order_date,status
1001,1,101,1,2026-10-01,Completed
1002,2,102,2,2026-10-02,Completed
1003,3,103,3,2026-10-03,Pending
1004,1,104,2,2026-10-04,Completed
1005,4,105,1,2026-10-05,Cancelled
1006,5,101,1,2026-10-06,Completed
1007,2,103,2,2026-10-07,Completed
1008,6,102,1,2026-10-08,Pending
1009,3,104,3,2026-10-09,Completed
1002,2,102,2,2026-10-02,Completed


In [0]:
# Create Bronze tables from raw DataFrames

customers_df.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_customers"
)

products_df.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_products"
)

orders_df.write.format("delta").mode("overwrite").saveAsTable(
    "bronze_orders"
)

print("Bronze Layer tables created successfully!")

Bronze Layer tables created successfully!


In [0]:
%sql
SELECT * FROM bronze_customers;

customer_id,customer_name,city
1,Arif,Kolkata
2,Priya,Delhi
3,Rahul,Mumbai
4,Sneha,Kolkata
5,Amit,Bangalore
2,Priya,Delhi
6,null,Hyderabad


In [0]:
%sql
SELECT * FROM bronze_products;

product_id,product_name,category,price
101,Laptop,Electronics,55000
102,Mobile,Electronics,20000
103,Headphones,Accessories,1500
104,Keyboard,Accessories,800
105,Monitor,Electronics,12000
103,Headphones,Accessories,1500


In [0]:
%sql
SELECT * FROM bronze_orders;

order_id,customer_id,product_id,quantity,order_date,status
1001,1,101,1,2026-10-01,Completed
1002,2,102,2,2026-10-02,Completed
1003,3,103,3,2026-10-03,Pending
1004,1,104,2,2026-10-04,Completed
1005,4,105,1,2026-10-05,Cancelled
1006,5,101,1,2026-10-06,Completed
1007,2,103,2,2026-10-07,Completed
1008,6,102,1,2026-10-08,Pending
1009,3,104,3,2026-10-09,Completed
1002,2,102,2,2026-10-02,Completed


In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")

products_df.write.format("delta").mode("overwrite").saveAsTable("bronze_products")

orders_df.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")

print("Bronze Layer created successfully!")

Bronze Layer created successfully!


In [0]:
print("Customers:", spark.table("bronze_customers").count())
print("Products:", spark.table("bronze_products").count())
print("Orders:", spark.table("bronze_orders").count())

Customers: 7
Products: 6
Orders: 10


In [0]:
from pyspark.sql.functions import col, trim

# Read raw data from Bronze Layer
customers_silver = spark.table("bronze_customers")

# Remove duplicate customer records
customers_silver = customers_silver.dropDuplicates(["customer_id"])

# Remove records with missing customer names
customers_silver = customers_silver.filter(
    col("customer_name").isNotNull()
)

# Remove extra spaces from customer names and cities
customers_silver = customers_silver.withColumn(
    "customer_name", trim(col("customer_name"))
).withColumn(
    "city", trim(col("city"))
)

# Save cleaned data to Silver Layer
customers_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver_customers"
)

display(customers_silver)

customer_id,customer_name,city
1,Arif,Kolkata
2,Priya,Delhi
3,Rahul,Mumbai
4,Sneha,Kolkata
5,Amit,Bangalore


In [0]:
from pyspark.sql.functions import col, trim

# Read products from Bronze Layer
products_silver = spark.table("bronze_products")

# Remove duplicate products
products_silver = products_silver.dropDuplicates(["product_id"])

# Remove invalid product names
products_silver = products_silver.filter(
    col("product_name").isNotNull()
)

# Remove extra spaces
products_silver = products_silver.withColumn(
    "product_name", trim(col("product_name"))
).withColumn(
    "category", trim(col("category"))
)

# Keep only valid prices
products_silver = products_silver.filter(
    col("price") > 0
)

# Save to Silver Layer
products_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver_products"
)

display(products_silver)

product_id,product_name,category,price
101,Laptop,Electronics,55000
102,Mobile,Electronics,20000
103,Headphones,Accessories,1500
104,Keyboard,Accessories,800
105,Monitor,Electronics,12000


In [0]:
from pyspark.sql.functions import col, to_date

# Read orders from Bronze Layer
orders_silver = spark.table("bronze_orders")

# Remove duplicate orders
orders_silver = orders_silver.dropDuplicates(["order_id"])

# Convert order_date to Date type
orders_silver = orders_silver.withColumn(
    "order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)

# Remove invalid records
orders_silver = orders_silver.filter(
    (col("order_id").isNotNull()) &
    (col("customer_id").isNotNull()) &
    (col("product_id").isNotNull()) &
    (col("quantity") > 0) &
    (col("order_date").isNotNull())
)

# Save cleaned data
orders_silver.write.format("delta").mode("overwrite").saveAsTable(
    "silver_orders"
)

display(orders_silver)

order_id,customer_id,product_id,quantity,order_date,status
1001,1,101,1,2026-10-01,Completed
1002,2,102,2,2026-10-02,Completed
1003,3,103,3,2026-10-03,Pending
1004,1,104,2,2026-10-04,Completed
1005,4,105,1,2026-10-05,Cancelled
1006,5,101,1,2026-10-06,Completed
1007,2,103,2,2026-10-07,Completed
1008,6,102,1,2026-10-08,Pending
1009,3,104,3,2026-10-09,Completed


In [0]:
from pyspark.sql.functions import col

# Read Silver tables
customers = spark.table("silver_customers")
products = spark.table("silver_products")
orders = spark.table("silver_orders")

# Join orders with customers and products
joined_df = (
    orders
    .join(customers, on="customer_id", how="inner")
    .join(products, on="product_id", how="inner")
)

# Calculate revenue for each order
joined_df = joined_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(joined_df)

product_id,customer_id,order_id,quantity,order_date,status,customer_name,city,product_name,category,price,revenue
104,1,1004,2,2026-10-04,Completed,Arif,Kolkata,Keyboard,Accessories,800,1600
101,1,1001,1,2026-10-01,Completed,Arif,Kolkata,Laptop,Electronics,55000,55000
103,3,1003,3,2026-10-03,Pending,Rahul,Mumbai,Headphones,Accessories,1500,4500
102,2,1002,2,2026-10-02,Completed,Priya,Delhi,Mobile,Electronics,20000,40000
105,4,1005,1,2026-10-05,Cancelled,Sneha,Kolkata,Monitor,Electronics,12000,12000
101,5,1006,1,2026-10-06,Completed,Amit,Bangalore,Laptop,Electronics,55000,55000
104,3,1009,3,2026-10-09,Completed,Rahul,Mumbai,Keyboard,Accessories,800,2400
103,2,1007,2,2026-10-07,Completed,Priya,Delhi,Headphones,Accessories,1500,3000


In [0]:
from pyspark.sql.functions import sum, count

product_sales = (
    joined_df
    .groupBy("product_id", "product_name", "category")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum("revenue").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)

display(product_sales)

product_id,product_name,category,total_quantity_sold,total_revenue,total_orders
101,Laptop,Electronics,2,110000,2
102,Mobile,Electronics,2,40000,1
105,Monitor,Electronics,1,12000,1
103,Headphones,Accessories,5,7500,2
104,Keyboard,Accessories,5,4000,2


In [0]:
product_sales.write.format("delta").mode("overwrite").saveAsTable(
    "gold_product_sales"
)

print("Gold Product Sales table created successfully!")

Gold Product Sales table created successfully!


In [0]:
from pyspark.sql.functions import col

# Read Silver tables
customers = spark.table("silver_customers")
products = spark.table("silver_products")
orders = spark.table("silver_orders")

# Join orders with customers and products
joined_df = (
    orders
    .join(customers, on="customer_id", how="inner")
    .join(products, on="product_id", how="inner")
)

# Calculate revenue for each order
joined_df = joined_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(joined_df)

product_id,customer_id,order_id,quantity,order_date,status,customer_name,city,product_name,category,price,revenue
104,1,1004,2,2026-10-04,Completed,Arif,Kolkata,Keyboard,Accessories,800,1600
101,1,1001,1,2026-10-01,Completed,Arif,Kolkata,Laptop,Electronics,55000,55000
103,3,1003,3,2026-10-03,Pending,Rahul,Mumbai,Headphones,Accessories,1500,4500
102,2,1002,2,2026-10-02,Completed,Priya,Delhi,Mobile,Electronics,20000,40000
105,4,1005,1,2026-10-05,Cancelled,Sneha,Kolkata,Monitor,Electronics,12000,12000
101,5,1006,1,2026-10-06,Completed,Amit,Bangalore,Laptop,Electronics,55000,55000
104,3,1009,3,2026-10-09,Completed,Rahul,Mumbai,Keyboard,Accessories,800,2400
103,2,1007,2,2026-10-07,Completed,Priya,Delhi,Headphones,Accessories,1500,3000


In [0]:
from pyspark.sql.functions import sum, count, col

product_sales = (
    joined_df
    .filter(col("status") == "Completed")
    .groupBy("product_id", "product_name", "category")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum("revenue").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)

display(product_sales)

product_id,product_name,category,total_quantity_sold,total_revenue,total_orders
101,Laptop,Electronics,2,110000,2
102,Mobile,Electronics,2,40000,1
104,Keyboard,Accessories,5,4000,2
103,Headphones,Accessories,2,3000,1


In [0]:
product_sales.write.format("delta").mode("overwrite").saveAsTable(
    "gold_product_sales"
)

print("Gold Product Sales table created successfully!")

Gold Product Sales table created successfully!


In [0]:
from pyspark.sql.functions import sum, count, col

product_sales = (
    joined_df
    .filter(col("status") == "Completed")
    .groupBy("product_id", "product_name", "category")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        sum("revenue").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)

display(product_sales)

product_id,product_name,category,total_quantity_sold,total_revenue,total_orders
101,Laptop,Electronics,2,110000,2
102,Mobile,Electronics,2,40000,1
104,Keyboard,Accessories,5,4000,2
103,Headphones,Accessories,2,3000,1


In [0]:
product_sales.write.format("delta").mode("overwrite").saveAsTable(
    "gold_product_sales"
)

print("Gold Product Sales table created successfully!")

Gold Product Sales table created successfully!


In [0]:
from pyspark.sql.functions import sum, count, col, coalesce, lit

customers = spark.table("silver_customers")
orders = spark.table("silver_orders")
products = spark.table("silver_products")

# Keep only completed orders
completed_orders = orders.filter(
    col("status") == "Completed"
)

# Join orders with products to calculate revenue
order_revenue = (
    completed_orders
    .join(products, on="product_id", how="inner")
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
)

# Calculate spending for each customer
spending = (
    order_revenue
    .groupBy("customer_id")
    .agg(
        sum("revenue").alias("total_spending"),
        count("order_id").alias("total_orders")
    )
)

# Keep all customers, including those with no completed orders
customer_spending = (
    customers
    .join(spending, on="customer_id", how="left")
    .withColumn(
        "total_spending",
        coalesce(col("total_spending"), lit(0))
    )
    .withColumn(
        "total_orders",
        coalesce(col("total_orders"), lit(0))
    )
    .orderBy("customer_id")
)

display(customer_spending)

customer_id,customer_name,city,total_spending,total_orders
1,Arif,Kolkata,56600,2
2,Priya,Delhi,43000,2
3,Rahul,Mumbai,2400,1
4,Sneha,Kolkata,0,0
5,Amit,Bangalore,55000,1


In [0]:
from pyspark.sql.functions import count, sum, col

order_summary = (
    spark.table("silver_orders")
    .groupBy("status")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity")
    )
)

display(order_summary)

status,total_orders,total_quantity
Completed,6,11
Pending,2,4
Cancelled,1,1


In [0]:
order_summary.write.format("delta").mode("overwrite").saveAsTable(
    "gold_order_summary"
)

print("Gold Order Summary created successfully!")

Gold Order Summary created successfully!


In [0]:
customer_spending.write.format("delta").mode("overwrite").saveAsTable(
    "gold_customer_spending"
)

print("Gold Customer Spending table updated successfully!")

Gold Customer Spending table updated successfully!


In [0]:
print("========== PROJECT VALIDATION ==========")

print("Bronze Customers:", spark.table("bronze_customers").count())
print("Bronze Products:", spark.table("bronze_products").count())
print("Bronze Orders:", spark.table("bronze_orders").count())

print("Silver Customers:", spark.table("silver_customers").count())
print("Silver Products:", spark.table("silver_products").count())
print("Silver Orders:", spark.table("silver_orders").count())

print("Gold Product Sales:", spark.table("gold_product_sales").count())
print("Gold Customer Spending:", spark.table("gold_customer_spending").count())
print("Gold Order Summary:", spark.table("gold_order_summary").count())

print("========== VALIDATION COMPLETED ==========")

========== PROJECT VALIDATION ==========
Bronze Customers: 7
Bronze Products: 6
Bronze Orders: 10
Silver Customers: 5
Silver Products: 5
Silver Orders: 9
Gold Product Sales: 4
Gold Customer Spending: 4
Gold Order Summary: 3
========== VALIDATION COMPLETED ==========


In [0]:
customer_spending.write.format("delta").mode("overwrite").saveAsTable(
    "gold_customer_spending"
)

print("Customer spending table updated!")

Customer spending table updated!


In [0]:
display(spark.table("gold_customer_spending"))

customer_id,customer_name,city,total_spending,total_orders
1,Arif,Kolkata,56600,2
2,Priya,Delhi,43000,2
3,Rahul,Mumbai,2400,1
4,Sneha,Kolkata,0,0
5,Amit,Bangalore,55000,1


In [0]:
print("Gold Product Sales:", spark.table("gold_product_sales").count())
print("Gold Customer Spending:", spark.table("gold_customer_spending").count())
print("Gold Order Summary:", spark.table("gold_order_summary").count())

Gold Product Sales: 4
Gold Customer Spending: 5
Gold Order Summary: 3


In [0]:
print("========== UPDATED FINAL VALIDATION ==========")

print("Bronze Customers:", spark.table("bronze_customers").count())
print("Bronze Products:", spark.table("bronze_products").count())
print("Bronze Orders:", spark.table("bronze_orders").count())

print("Silver Customers:", spark.table("silver_customers").count())
print("Silver Products:", spark.table("silver_products").count())
print("Silver Orders:", spark.table("silver_orders").count())

print("Gold Product Sales:", spark.table("gold_product_sales").count())
print("Gold Customer Spending:", spark.table("gold_customer_spending").count())
print("Gold Order Summary:", spark.table("gold_order_summary").count())

print("========== VALIDATION COMPLETED ==========")

========== UPDATED FINAL VALIDATION ==========
Bronze Customers: 7
Bronze Products: 6
Bronze Orders: 10
Silver Customers: 5
Silver Products: 5
Silver Orders: 9
Gold Product Sales: 4
Gold Customer Spending: 5
Gold Order Summary: 3
========== VALIDATION COMPLETED ==========
